## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 01 - Prepare Original, Full-PNG, and YOLO ROI Datasets |
| Model / workflow | DenseNet-121/201 |
| Input | 224x224 |
| Loss | not reported |
| Training / pipeline | YOLO detection/ROI workflow |
| Result | Full bilateral HDF5 source: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/DetKneeData/H5 |


# 01 - Prepare Original, Full-PNG, and YOLO ROI Datasets

Run this notebook once. It uses the one existing `KneeXrayData.zip` archive, creates full bilateral PNG images, then produces a resumable square-ROI dataset with `train`, `val`, and `test` splits.

It does not train a classifier. The original 224x224 published images are retained in place; they are not copied or modified.


In [ ]:
!pip -q install "ultralytics>=8.3,<9" "h5py>=3.9"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:0000:01


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import math
import os
import time
from pathlib import Path
from zipfile import ZipFile

import cv2
import h5py
import numpy as np
import torch
from ultralytics import YOLO


Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Extract the source archive only when required

Both source folders are required: the published 224x224 PNGs provide KL labels, while the HDF5 files contain full bilateral radiographs.


In [ ]:
ARCHIVE = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip")

# extract dir: contains the extracted files (KneeXrayData/ClsKLData, KneeXrayData/DetKneeData)
# ClsKLData: contains the published 224x224 PNGs (kneeKL224)
# DetKneeData: contains the full bilateral HDF5 files (H5) 
# H5 is the full bilateral radiographs -> convert to PNGs
EXTRACT_DIR = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted")

# data root: contains the extracted files (ClsKLData/kneeKL224, DetKneeData/H5)
DATA_ROOT = EXTRACT_DIR / "KneeXrayData"

# original root: contains the published 224x224 PNGs (ClsKLData/kneeKL224)
ORIGINAL_ROOT = DATA_ROOT / "ClsKLData/kneeKL224"

# H5 root: contains the full bilateral HDF5 files (DetKneeData/H5)
H5_ROOT = DATA_ROOT / "DetKneeData/H5"

# if not exists, raise error
if not ARCHIVE.is_file():
    raise FileNotFoundError(f"ZIP file not found: {ARCHIVE}")

# if not exists, create the extract dir
if not ORIGINAL_ROOT.is_dir() or not H5_ROOT.is_dir():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ARCHIVE) as archive:
        archive.extractall(EXTRACT_DIR)

# check if the original and H5 roots exist
for required in (ORIGINAL_ROOT, H5_ROOT):
    if not required.is_dir():
        raise FileNotFoundError(f"Required source folder not found: {required}")

print("Original 224x224 dataset:", ORIGINAL_ROOT)
print("Full bilateral HDF5 source:", H5_ROOT)


Original 224x224 dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224
Full bilateral HDF5 source: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/DetKneeData/H5


## Export full bilateral radiographs as PNG

Each bilateral image is saved once by patient ID. It cannot be placed in a single KL-grade folder because its left and right knees can have different grades. This cell resumes safely by skipping PNGs already written.


In [ ]:
# Ouput of full png which converted from h5 files
FULL_PNG_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2"
)
SPLITS = ("train", "val", "test")

for split in SPLITS:

    # define source and output directories
    source_dir = H5_ROOT / f"{split}H5"
    output_dir = FULL_PNG_ROOT / split

    # if the output directory already exists, skip this split
    if not source_dir.is_dir():
        raise FileNotFoundError(source_dir)

    #else
    output_dir.mkdir(parents=True, exist_ok=True)
    h5_paths = sorted(source_dir.glob("*.h5"))


    for index, h5_path in enumerate(h5_paths, start=1):

        # skip if the output PNG already exists
        output_path = output_dir / f"{h5_path.stem}.png"
        if output_path.is_file():
            continue

        # read the HDF5 file and convert to a full PNG image
        with h5py.File(h5_path, "r") as handle:

            # read the image from the HDF5 file -> numpy array  
            image = np.asarray(handle["images"])

        # if the image is grayscale, convert it to RGB by repeating the single channel
        if image.ndim == 2:

            # repeat the single channel to create an RGB image
            image = np.repeat(image[..., None], 3, axis=2)

        # if the image has a single channel, repeat it to create an RGB image

        # shape[-1] is the number of channels
        if image.shape[-1] == 1:
            image = np.repeat(image, 3, axis=2)

        # clip: limit the values to the range [0, 255] -> convert to uint8
        image = np.clip(image[..., :3], 0, 255).astype(np.uint8)

        # write the image to the output path
        if not cv2.imwrite(str(output_path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR)):
            raise RuntimeError(f"Cannot write {output_path}")

        # print the progress
        # index % 100 == 0: print every 100 images
        # index == len(h5_paths): print the last image
        if index % 100 == 0 or index == len(h5_paths):
            print(f"{split}: {index}/{len(h5_paths)} full PNG images")


train: 100/2889 full PNG images
train: 200/2889 full PNG images
train: 300/2889 full PNG images
train: 400/2889 full PNG images
train: 500/2889 full PNG images
train: 600/2889 full PNG images
train: 700/2889 full PNG images
train: 800/2889 full PNG images
train: 900/2889 full PNG images
train: 1000/2889 full PNG images
train: 1100/2889 full PNG images
train: 1200/2889 full PNG images
train: 1300/2889 full PNG images
train: 1400/2889 full PNG images
train: 1500/2889 full PNG images
train: 1600/2889 full PNG images
train: 1700/2889 full PNG images
train: 1800/2889 full PNG images
train: 1900/2889 full PNG images
train: 2000/2889 full PNG images
train: 2100/2889 full PNG images
train: 2200/2889 full PNG images
train: 2300/2889 full PNG images
train: 2400/2889 full PNG images
train: 2500/2889 full PNG images
train: 2600/2889 full PNG images
train: 2700/2889 full PNG images
train: 2800/2889 full PNG images
train: 2889/2889 full PNG images
val: 100/413 full PNG images
val: 200/413 full PNG i

## Create the square YOLO ROI dataset

The source boxes are expanded by 1.15x, then the shorter dimension is extended to form a square. Padding is used only where that square would cross the source-image boundary. No center crop is used. Existing ROIs are skipped, so this cell resumes after a Colab interruption.


In [ ]:
# load the yolov8 model checkpoint
YOLO_CHECKPOINT = Path("/content/drive/MyDrive/Models/yolov8_checkpoint/best.pt")

# output of square roi dataset
ROI_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/"
    "densenet121_yolo_square_roi_trainvaltest_v2"
)

# config: batch size, confidence threshold, image size (640x640)
YOLO_BATCH_SIZE = 32
YOLO_CONFIDENCE = 0.45
YOLO_IMAGE_SIZE = 640

# YOLO bounding boxes are expanded by this factor to create square ROIs 
# for example, a box of 100x200 pixels will be expanded to 230x230 pixels if BOX_EXPANSION=1.15
BOX_EXPANSION = 1.15

# check that the YOLO checkpoint exists
if not YOLO_CHECKPOINT.is_file():
    raise FileNotFoundError(YOLO_CHECKPOINT)

labels = {}

# loop each split (train,val,test)
for split in SPLITS:

    # loop each kl (0 -> 4)
    for grade in range(5):

        # get labels of each image
        for path in (ORIGINAL_ROOT / split / str(grade)).glob("*.png"):
            # path.stem[:-1] is the patient ID without the side (R or L)
            # path.stem[-1].upper() is the side (R or L)
            # example of file name: patient123R.png
            labels[(split, path.stem[:-1], path.stem[-1].upper())] = grade



# function to add padding to make a square ROI from a YOLO bounding box
def make_square_roi(image, box):
    # image.shape[:2] is the height and width of the image
    # image.shape[2] is the number of channels
    height, width = image.shape[:2]
    
    # box is the bounding box of the ROI
    # box is a list of 4 values: [x1, y1, x2, y2]
    # x1, y1 is the top-left corner of the bounding box
    # x2, y2 is the bottom-right corner of the bounding box
    x1, y1, x2, y2 = map(float, box)

    # calc height + width of box
    box_width, box_height = x2 - x1, y2 - y1

    if box_width <= 0 or box_height <= 0:
        raise ValueError(f"Invalid YOLO box: {box}")

    # calc center of x, y
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2

    # expand to 1.15
    side = int(math.ceil(max(box_width, box_height) * BOX_EXPANSION))

    # get new x1,y1,x2,y2
    wanted_x1 = int(math.floor(center_x - side / 2))
    wanted_y1 = int(math.floor(center_y - side / 2))
    wanted_x2, wanted_y2 = wanted_x1 + side, wanted_y1 + side
    crop = image[max(0, wanted_y1):min(height, wanted_y2), max(0, wanted_x1):min(width, wanted_x2)]
    if crop.size == 0:
        raise RuntimeError(f"Empty ROI crop for {box}")

    # add padding to make a square ROI
    # max(0, -wanted_y1): add padding to the top
    # max(0, wanted_y2 - height): add padding to the bottom
    # max(0, -wanted_x1): add padding to the left
    # max(0, wanted_x2 - width): add padding to the right
    return cv2.copyMakeBorder(
        crop,
        max(0, -wanted_y1), max(0, wanted_y2 - height),
        max(0, -wanted_x1), max(0, wanted_x2 - width),
        cv2.BORDER_CONSTANT, value=(0, 0, 0),
    )


# load yolo model 
detector = YOLO(str(YOLO_CHECKPOINT))
detector_device = 0 if torch.cuda.is_available() else "cpu"

# loop each split (train,val,test)
for split in SPLITS:
    for grade in range(5):
        (ROI_ROOT / split / str(grade)).mkdir(parents=True, exist_ok=True)

    # get all full PNG images for this split and check which ones are missing ROIs
    image_paths = sorted((FULL_PNG_ROOT / split).glob("*.png"))
    pending = []


    # check which images are missing ROIs for both knees
    for image_path in image_paths:

        # get the patient ID from the image filename 
        # image_path.stem is the filename without the extension, e.g., "patient123R" or "patient123L"
        patient = image_path.stem

        # check if the ROIs for both knees (R and L) exist for this patient
        required = [ROI_ROOT / split / str(labels[(split, patient, side)]) / f"{patient}{side}.png" for side in ("R", "L")]

        # if any(not path.is_file() for path in required):
        if not all(path.is_file() for path in required):
            pending.append(image_path)

    # loop through the pending images in batches and run YOLO detection
    for start in range(0, len(pending), YOLO_BATCH_SIZE):

        # get the batch of image paths and read the images
        batch_paths = pending[start:start + YOLO_BATCH_SIZE]

        # read the images
        batch_images = [cv2.imread(str(path), cv2.IMREAD_COLOR) for path in batch_paths]

        # check if any image failed to load
        if any(image is None for image in batch_images):
            bad = batch_paths[next(i for i, image in enumerate(batch_images) if image is None)]
            raise RuntimeError(f"Cannot read full PNG: {bad}")

        # run YOLO detection on the batch of images
        predictions = detector.predict(
            source=batch_images, conf=YOLO_CONFIDENCE, imgsz=YOLO_IMAGE_SIZE,
            device=detector_device, batch=YOLO_BATCH_SIZE, save=False, verbose=False,
        )

        # loop through each image in the batch and process the YOLO predictions
        for image_path, image, prediction in zip(batch_paths, batch_images, predictions):

            # get the patient ID from the image filename
            patient = image_path.stem

            # get the YOLO bounding boxes and confidence scores, sort by score, and keep the top 2 boxes
            # for example: [[x1, y1, x2, y2], [x1, y1, x2, y2] ]
            boxes = prediction.boxes.xyxy.detach().cpu().numpy()

            # get the confidence scores
            # for example: [0.9, 0.8]
            scores = prediction.boxes.conf.detach().cpu().numpy()

            # sort the boxes by confidence score in descending order and keep the top 2 boxes
            # for example: [[x1, y1, x2, y2], [x1, y1, x2, y2] ]
            # ::-1: reverse the order: high -> low
            # [:2]: keep the top 2 boxes
            boxes = boxes[np.argsort(scores)[::-1][:2]]

            # sort the boxes by the x1 + x2 in ascending order
            # for example: [[x1, y1, x2, y2], [x1, y1, x2, y2] ]
            # key=lambda box: float(box[0] + box[2]): sort by the x1 + x2
            boxes = sorted(boxes, key=lambda box: float(box[0] + box[2]))

            # if the number of boxes is not equal to 2, raise an error
            if len(boxes) != 2:
                raise RuntimeError(f"Expected two knee detections: {image_path}")

            # loop through the boxes and save the square ROIs for each knee (R and L)
            # first box map to R (right knee)
            # second box map to L (left knee)
            for box, side in zip(boxes, ("R", "L")):

                # get the KL grade for this patient and side from the labels dictionary
                grade = labels.get((split, patient, side))

                # if the grade is None, raise an error
                if grade is None:
                    raise RuntimeError(f"Missing label: {split}/{patient}{side}")

                # create the destination path for the ROI image
                destination = ROI_ROOT / split / str(grade) / f"{patient}{side}.png"
                if not destination.is_file() and not cv2.imwrite(str(destination), make_square_roi(image, box)):
                    raise RuntimeError(f"Cannot write {destination}")
        print(f"{split}: {min(start + len(batch_paths), len(pending))}/{len(pending)} pending bilateral images")

print("ROI dataset ready:", ROI_ROOT)


train: 32/2889 pending bilateral images
train: 64/2889 pending bilateral images
train: 96/2889 pending bilateral images
train: 128/2889 pending bilateral images
train: 160/2889 pending bilateral images
train: 192/2889 pending bilateral images
train: 224/2889 pending bilateral images
train: 256/2889 pending bilateral images
train: 288/2889 pending bilateral images
train: 320/2889 pending bilateral images
train: 352/2889 pending bilateral images
train: 384/2889 pending bilateral images
train: 416/2889 pending bilateral images
train: 448/2889 pending bilateral images
train: 480/2889 pending bilateral images
train: 512/2889 pending bilateral images
train: 544/2889 pending bilateral images
train: 576/2889 pending bilateral images
train: 608/2889 pending bilateral images
train: 640/2889 pending bilateral images
train: 672/2889 pending bilateral images
train: 704/2889 pending bilateral images
train: 736/2889 pending bilateral images
train: 768/2889 pending bilateral images
train: 800/2889 pen

## Confirm Drive writes before optional shutdown

This is a practical mounted-Drive check: it waits until file count and total size are stable for 60 seconds, calls `os.sync()`, then optionally powers off the Colab VM. Set `AUTO_SHUTDOWN` to `True` only in the last run of this notebook.


In [ ]:

WATCHED_DIRS = [FULL_PNG_ROOT, ROI_ROOT]
POLL_SECONDS = 10
STABLE_SECONDS = 60
AUTO_SHUTDOWN = False

# function to count the number of files and total size in a directory
def snapshot(path):
    files = [file_path for file_path in path.rglob("*") if file_path.is_file()]
    return len(files), sum(file_path.stat().st_size for file_path in files)


previous = None
stable_since = None

# loop until the watched directories are stable for STABLE_SECONDS
while True:

    # take a snapshot of the current state of the watched directories
    current = {str(path): snapshot(path) for path in WATCHED_DIRS}
    print(current)

    # if the current snapshot is the same as the previous snapshot, check if it has been stable for STABLE_SECONDS
    if current == previous:
        stable_since = stable_since or time.time()
        if time.time() - stable_since >= STABLE_SECONDS:
            break

    # if the current snapshot is different from the previous snapshot, reset the stable_since timer
    else:
        previous, stable_since = current, None
    time.sleep(POLL_SECONDS)

os.sync()
print("Drive outputs are stable and sync has been requested.")
if AUTO_SHUTDOWN:
    print("Shutting down Colab in 15 seconds...")
    time.sleep(15)
    os.system("sudo shutdown -h now")


{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/Kn